In [1]:
suppressMessages({
    library(Seurat)
    library(dplyr)
    library(pvclust)
    library(Matrix)
})

In [2]:
# read input RDS, and for loop it
path.list <- read.table('0.glia.metagenes.chordate.path')
id.list <- c()
for (i in 1:nrow(path.list)){
    abc = strsplit(path.list[i,1], split = "/")
    id <- abc[[1]][lengths(abc)]
    del_id <- path.list[i,2]
    assign(del_id, readRDS(path.list[i,1]))
    id.list <- c(id.list, del_id)
}

# build list containing objects
rds.list <- lapply(id.list, get)
print(id.list)
names(rds.list) <- id.list

[1] "Bflo" "Hsap" "Mmus" "Pmar" "Pvit"


In [3]:
genes <- lapply(rds.list, FUN = rownames)
HVG <- lapply(rds.list, FUN = VariableFeatures)

In [4]:
genes <- names(table(unlist(genes)))[table(unlist(genes)) == 5]
HVG <- names(table(unlist(HVG)))[table(unlist(HVG)) >= 3 ]
TF <- read.delim('../0.prep_data/orthogroups.5chordates.TFs.txt', header = F)$V1
genes_HVG <- genes[genes %in% HVG]
genes_TF <- genes[genes %in% TF]
genes_HVG_TF <- intersect(genes_HVG, genes_TF)

In [5]:
SI <- lapply(X = rds.list, FUN = function(obj){
    obj@meta.data$Refined.family <- paste(obj@meta.data$Species, obj@meta.data$`Refined family`, sep = '_')
    sampled_df <- obj@meta.data
    sampled_df$cellname <- rownames(sampled_df)
    sampled_df <- sampled_df %>% group_by(Refined.family) %>% slice_sample(n = 500) %>% ungroup()
    obj <- subset(obj, cells = as.character(sampled_df$cellname))
    
    norm.mat <- GetAssayData(obj, assay = "SCT", slot  = "data")
    celltypes <- unique(obj@meta.data$Refined.family)
    
    mean.by.celltype <- sapply(celltypes, function(ct) {
        cells <- colnames(obj)[which(obj@meta.data$Refined.family == ct)]
        Matrix::rowMeans(norm.mat[, cells, drop = FALSE])
    })
    # Convert to matrix (metagene × celltypes)
    mean.by.celltype <- as.matrix(mean.by.celltype)

    # Calculate global mean per gene (across all cells)
    mean.allcells <- Matrix::rowMeans(norm.mat)

    # Specificity index
    specificity.index <- sweep(mean.by.celltype, 1, mean.allcells, "/")
})

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


In [6]:
SI2_HVG <- cbind(SI[['Hsap']][genes_HVG, ], SI[['Mmus']][genes_HVG, ], 
                 SI[['Pvit']][genes_HVG, ], SI[['Pmar']][genes_HVG, ], SI[['Bflo']][genes_HVG, ])
SI2_TF <- cbind(SI[['Hsap']][genes_TF, ], SI[['Mmus']][genes_TF, ], 
                SI[['Pvit']][genes_TF, ], SI[['Pmar']][genes_TF, ], SI[['Bflo']][genes_TF, ])
SI2_TF_HVG <- cbind(SI[['Hsap']][genes_HVG_TF, ], SI[['Mmus']][genes_HVG_TF, ], 
                SI[['Pvit']][genes_HVG_TF, ], SI[['Pmar']][genes_HVG_TF, ], SI[['Bflo']][genes_HVG_TF, ])

In [ ]:
set.seed(42)

pdf('chordate_glia_tree_on_metagene_HVG.pdf', width = 10, height = 6)
# remove the non-glia population
pv <- pvclust(SI2_HVG[, !(colnames(SI2_HVG) %in% c('Bflo_22', 'Bflo_18', 'Bflo_19'))], nboot=1000, parallel=TRUE,
              method.hclust = "average", method.dist = function(z){as.dist(1-cor(z,use="pa",method="spearman"))})
plot(pv)
dev.off()

pdf('chordate_glia_tree_on_metagene_TF.pdf', width = 10, height = 6)
pv <- pvclust(SI2_TF[, !(colnames(SI2_TF) %in% c('Bflo_22', 'Bflo_18', 'Bflo_19'))], nboot=1000, parallel=TRUE,
              method.hclust = "average", method.dist = function(z){as.dist(1-cor(z,use="pa",method="spearman"))})
plot(pv)
dev.off()

pdf('chordate_glia_tree_on_metagene_HVG_TF.pdf', width = 10, height = 6)
pv <- pvclust(SI2_TF_HVG[, !(colnames(SI2_TF_HVG) %in% c('Bflo_22', 'Bflo_18', 'Bflo_19'))], nboot=1000, parallel=TRUE,
              method.hclust = "average", method.dist = function(z){as.dist(1-cor(z,use="pa",method="spearman"))})
plot(pv)
dev.off()

Creating a temporary cluster...done:
socket cluster with 71 nodes on host ‘localhost’
Multiscale bootstrap... Done.


pdf 
  2

Creating a temporary cluster...done:
socket cluster with 71 nodes on host ‘localhost’
Multiscale bootstrap... Done.


pdf 
  2

Creating a temporary cluster...done:
socket cluster with 71 nodes on host ‘localhost’
Multiscale bootstrap... Done.


pdf 
  2